# Introduction

This notebook aims to functionally characterize VMPFC by performing a meta-analytic decoding analysis using the Neurosynth database.

In [2]:
from nsplus.src.datasetplus import DatasetPlus
from nsplus.src.ranking import rank_terms
from pathlib import Path
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PATH = Path('../data')
PLOTS_PATH = Path('../plots')
RESULTS_PATH = Path('../results')

# ROI-Based Term Ranking
First, we apply a predefined spatial mask of the VMPFC (`VMPFC_mask_v2.nii`) to the comprehensive meta-analytic database. Using the `NSPlus` analytical module, we calculate the statistical associations between the activation of this specific ROI and all available psychological/cognitive terms. The terms are then quantitatively ranked based on their posterior probability (specifically, `pFgA_given_pF=0.50`), providing a raw, data-driven profile of the region's functional engagement.

In [4]:
csv_path = RESULTS_PATH / "NSPlus_rank/VMPFC.csv"
if not csv_path.exists():
    dataset = DatasetPlus.load_default_database()
    roi_path = str((DATA_PATH / 'masks/VMPFC_mask_v2.nii').resolve())
    dataset.mask(roi_path)
    csv_out = str(csv_path.resolve())
    info_df, rank_df = rank_terms(
        dataset=dataset, rank_by="pFgA_given_pF=0.50", csv_name=csv_out,
        extra_expr=(), ascending=False, rank_first=False, extra_info=[],)

# Targeted Functional Decoding
Because the raw Neurosynth vocabulary contains numerous anatomical, methodological, or irrelevant terms, we apply a targeted filtering strategy. We cross-reference the raw ranked list against a curated, hypothesis-driven vocabulary (`../data/neurosynth_data/term_selection.xls`).

Specifically, we restrict our decoding analysis to terms categorized under distinct psychological domains relevant to our research:
* **Social**
* **Valuation**
*  **Affect**
* **Others** (unspecified functions, for example, memory)

By intersecting the data-driven rankings with our curated vocabulary, we extract the top-performing features (the Top 15 and Top 50 most highly associated terms). These refined lists are subsequently exported for downstream visualization and theoretical interpretation.

In [ ]:
select_type_list = ['social', 'valuation', 'affect', 'others']
term_select_df = pd.read_excel(DATA_PATH / 'neurosynth_data/term_selection.xls').query('type in @select_type_list')
term_list = term_select_df['term'].tolist()
result_df = pd.read_csv(RESULTS_PATH / "NSPlus_rank/VMPFC.csv", index_col=0, skiprows=2)
result_df = result_df.query('term in @term_list')

# check top 15 and top 50 terms
top_15_df = result_df.iloc[:15]
top_15_df.to_csv(RESULTS_PATH / 'top15_term.csv', index=False)
top_15_df

In [ ]:
top_50_df = result_df.iloc[:50]
top_50_df.to_csv(RESULTS_PATH / 'top50_term.csv', index=False)
top_50_df